# Working with Unstructured Meshes

Copyright (C) 2020 Andreas Kloeckner

<details>
<summary>MIT License</summary>
Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.
</details>

This demo uses [meshmode](https://documen.tician.de/meshmode/).

In [16]:
import numpy as np
import numpy.linalg as la
import pyopencl as cl
import pyopencl.clmath
import meshmode.mesh.generation as mgen
from meshmode.array_context import PyOpenCLArrayContext
from meshmode.discretization import Discretization
from meshmode.discretization.poly_element import PolynomialRecursiveNodesGroupFactory

import matplotlib.pyplot as plt

In [6]:
cl_ctx = cl.create_some_context(interactive=True)
queue = cl.CommandQueue(cl_ctx)
actx = PyOpenCLArrayContext(queue)

In [3]:
order = 10

In [18]:
mesh = mgen.generate_sphere(r=1, order=order, uniform_refinement_rounds=1)

discr = Discretization(actx, mesh, PolynomialRecursiveNodesGroupFactory(order, "lgl"))

In [20]:
from meshmode.discretization.visualization import make_visualizer
vis = make_visualizer(actx, discr)

x, y, z = discr.nodes()
val = actx.np.sin(3*actx.thaw(x))

plt.figure(figsize=(10, 7))
vis.show_scalar_in_matplotlib_3d(val, do_show=False)

In [29]:
print(val[0].shape)
print(discr.nodes()[0][0].shape)

In [30]:
plt.plot(grp.unit_nodes[0], grp.unit_nodes[1], "o")
plt.gca().set_aspect("equal")

In [36]:
from modepy import multi_vandermonde

grp_basis = grp.basis_obj()
vdm = vandermonde(grp_basis.functions, grp.unit_nodes)
dr_vdm, ds_vdm = multi_vandermonde(grp_basis.gradients, grp.unit_nodes)

# Use la.solve in real code!
dr = dr_vdm @ la.inv(vdm)
ds = ds_vdm @ la.inv(vdm)

In [43]:
nodes = np.array([dof_ary[0] for dof_ary in actx.to_numpy(discr.nodes())])
dr_nodes = np.einsum("ij,dkj->dki", dr, nodes)
ds_nodes = np.einsum("ij,dkj->dki", ds, nodes)

normals = np.cross(dr_nodes, ds_nodes, axisa=0, axisb=0, axisc=0)

area_elements = la.norm(normals, axis=0)

area = np.einsum("i, ki->", grp.quadrature_rule().weights, area_elements)

true_area = 4*np.pi
abs(area - true_area)/true_area